# Notebook 05 — Backtest & Trading Simulation

**Experiment 5** from the proposal.

We simulate a simple long/short strategy on **out-of-sample** data (2023–2024):

| Signal | Action |
|--------|--------|
| Bullish pattern detected (conf ≥ 70 %) | Go **long** for 5 trading days |
| Bearish pattern detected (conf ≥ 70 %) | Go **short** for 5 trading days |
| No-pattern class | No trade |

Performance metrics:
- **Directional accuracy** (vs 50 % baseline)
- **Sharpe ratio** (risk-adjusted return)
- **Cumulative P&L** curve
- **Benchmark comparison** vs buy-and-hold S&P 500


In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from config import *
from src.models.cnn               import PatternCNN
from src.verification.stock_data  import run_verification, SP500_TICKERS
from src.verification.backtest    import (
    trading_simulation, print_summary, per_class_stats, sharpe_ratio
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load model & run out-of-sample detection (2023–2024)

In [ ]:
CKPT = os.path.join(ROOT, 'checkpoints', 'cnn_best.pt')
model = PatternCNN(num_classes=7)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE)['model_state_dict'])
model.to(DEVICE).eval()

oos_df = run_verification(
    model, DEVICE,
    tickers              = SP500_TICKERS,
    start                = BACKTEST_START,
    end                  = BACKTEST_END,
    window_size          = WINDOW_SIZE,
    step_size            = STEP_SIZE_SLIDE,
    confidence_threshold = CONFIDENCE_THRESHOLD,
    forward_days         = FORWARD_DAYS,
    verbose              = True,
)
oos_df.to_csv(os.path.join(ROOT, 'results', 'logs', 'backtest_detections.csv'), index=False)
print(f'Out-of-sample detections: {len(oos_df):,}')

## 2. Simulate the trading strategy

In [ ]:
fwd_col = f'forward_return_{FORWARD_DAYS}d'

results = trading_simulation(oos_df, forward_col=fwd_col, initial_cap=10_000)
print_summary(results)

## 3. Equity curve vs buy-and-hold benchmark

In [ ]:
# Download SPY as buy-and-hold benchmark
spy = yf.download('SPY', start=BACKTEST_START, end=BACKTEST_END,
                  auto_adjust=True, progress=False)['Close']
spy_norm = spy / spy.iloc[0] * 10_000   # start at same $10 000

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=False)

# ── Equity curve ──────────────────────────────────────────────────────────────
eq = results.get('equity_curve')
if eq is not None and len(eq) > 0:
    axes[0].plot(range(len(eq)), eq.values, label='Pattern strategy', color='steelblue', lw=2)
    axes[0].axhline(10_000, color='grey', linestyle='--', lw=1, label='Starting capital')
    axes[0].set_ylabel('Portfolio value ($)')
    axes[0].set_xlabel('Trade number')
    axes[0].set_title('Cumulative P&L — chart-pattern strategy (2023–2024)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No trades executed', ha='center', va='center',
                 transform=axes[0].transAxes, fontsize=14)

# ── SPY benchmark ─────────────────────────────────────────────────────────────
axes[1].plot(spy_norm.index, spy_norm.values, label='SPY (buy & hold)', color='orange', lw=2)
axes[1].set_ylabel('Portfolio value ($)')
axes[1].set_xlabel('Date')
axes[1].set_title('SPY Buy-and-Hold Benchmark (2023–2024)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'figures', 'equity_curve.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 4. Per-class backtesting accuracy (out-of-sample)

In [ ]:
pc = per_class_stats(oos_df, forward_col=fwd_col)
print('Per-class out-of-sample stats:')
print(pc[['n', 'directional_accuracy', 'mean_return', 'p_value']].to_string())

pc.to_csv(os.path.join(ROOT, 'results', 'logs', 'per_class_backtest.csv'))

## 5. Distribution of trade returns

In [ ]:
if results.get('n_trades', 0) > 0:
    rets_pct = results['trade_returns'] * 100

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(rets_pct, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
    axes[0].axvline(0, color='red', linestyle='--', label='Break-even')
    axes[0].axvline(np.mean(rets_pct), color='green', linestyle='-', label=f'Mean = {np.mean(rets_pct):.2f}%')
    axes[0].set_xlabel('Trade return (%)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of trade returns')
    axes[0].legend()

    # Cumulative win rate
    sorted_rets = np.sort(rets_pct)
    cum_frac    = np.linspace(0, 1, len(sorted_rets))
    axes[1].plot(sorted_rets, cum_frac, color='steelblue')
    axes[1].axvline(0, color='red', linestyle='--', label='Break-even')
    axes[1].set_xlabel('Return (%)')
    axes[1].set_ylabel('Cumulative fraction')
    axes[1].set_title('Cumulative distribution of trade returns')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, 'results', 'figures', 'return_distribution.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Win rate:          {(rets_pct > 0).mean():.2%}')
    print(f'Mean return/trade: {np.mean(rets_pct):.3f} %')
    print(f'Std return/trade:  {np.std(rets_pct):.3f} %')
    print(f'Sharpe ratio:      {results["sharpe"]:.3f}')